# Step 2. Feature Engineering (파생 변수 생성)

Step1에서 수집한 Tier 1 원시 데이터와 기존 프로젝트의 prices/macro를 결합하여  
**15개 파생 변수**를 생성하고, 이후 분석의 핵심 입력인 `df_reg_v2`를 구축합니다.

| 카테고리 | 파생 변수 | 개수 | 설명 |
|----------|----------|------|------|
| VIX 기간구조 | contango, slope_9d_3m, slope_3m_6m | 3개 | 변동성 곡면의 기울기와 형태 |
| 꼬리 위험 | SKEW_level, SKEW_zscore | 2개 | 블랙스완 우려 수준 및 Z-score |
| 실물 경기 | Cu_Au_ratio, Cu_Au_ratio_chg | 2개 | 경기 사이클 프록시 |
| 신용 시장 | HY_spread, HY_spread_chg, yield_curve, yield_curve_inv | 4개 | 신용 스트레스 + 곡선 역전 |
| 주간 매크로 | claims_4wma, claims_zscore, WEI_level, sahm_indicator | 4개 | 경기 Nowcasting |

In [1]:
# ============================================================
# 라이브러리 및 데이터 로드
# ============================================================
import pandas as pd
import numpy as np
import os, warnings
warnings.filterwarnings('ignore')

# ── 경로 설정 ──
BASE_DIR = os.path.dirname(os.getcwd())   # PJ12_Asset_Simulater/
DATA_DIR = os.path.join(os.getcwd(), 'data')

# ── 기존 프로젝트 데이터 ──
df_prices = pd.read_csv(os.path.join(BASE_DIR, 'prices.csv'),
                        index_col=0, parse_dates=True)
df_macro  = pd.read_csv(os.path.join(BASE_DIR, 'macro.csv'),
                        index_col=0, parse_dates=True)

# ── Step1 대안데이터 ──
df_alt = pd.read_csv(os.path.join(DATA_DIR, 'alt_tier1.csv'),
                     index_col=0, parse_dates=True)

print(f'prices  : {df_prices.shape}')
print(f'macro   : {df_macro.shape}')
print(f'alt_tier1: {df_alt.shape}')
print(f'alt 컬럼 : {list(df_alt.columns)}')

prices  : (1825, 13)
macro   : (1320, 3)
alt_tier1: (1304, 10)
alt 컬럼 : ['^VIX9D', '^VIX3M', '^VIX6M', '^SKEW', 'HG=F', 'BAMLH0A0HYM2', 'T10Y2Y', 'ICSA', 'WEI', 'SAHMREALTIME']


## 2-1. VIX 기간구조 파생 변수 (3개)

- **VIX_contango**: VIX3M / VIX - 1 (양수 = 정상 콘탱고, 음수 = 백워데이션 = 위기 임박)  
- **VIX_slope_9d_3m**: VIX3M - VIX9D (단기-중기 기울기)  
- **VIX_slope_3m_6m**: VIX6M - VIX3M (중기-장기 기울기)

In [2]:
# ============================================================
# VIX 기간구조 파생 변수
# ============================================================

# 기존 프로젝트의 ^VIX (현물 VIX) 가져오기
vix_spot = df_prices['^VIX'].reindex(df_alt.index).ffill().bfill()

# (1) 콘탱고/백워데이션 비율
#     VIX3M / VIX_spot - 1
#     양수 → 선물이 현물보다 높음 (정상, 시장 안정)
#     음수 → 선물이 현물보다 낮음 (백워데이션, 위기 임박)
df_feat = pd.DataFrame(index=df_alt.index)
df_feat['VIX_contango'] = df_alt['^VIX3M'] / vix_spot - 1

# (2) 단기-중기 기울기: VIX3M - VIX9D
#     기울기 양수 = 정상적 우상향, 음수 = 단기가 중기 초과 (급성 스트레스)
df_feat['VIX_slope_9d_3m'] = df_alt['^VIX3M'] - df_alt['^VIX9D']

# (3) 중기-장기 기울기: VIX6M - VIX3M
#     기울기 축소 = 장기 변동성도 상승 중 (구조적 리스크)
df_feat['VIX_slope_3m_6m'] = df_alt['^VIX6M'] - df_alt['^VIX3M']

print('=== VIX 기간구조 파생 변수 ===')
df_feat[['VIX_contango', 'VIX_slope_9d_3m', 'VIX_slope_3m_6m']].describe().round(4)

=== VIX 기간구조 파생 변수 ===


,VIX_contango,VIX_slope_9d_3m,VIX_slope_3m_6m
count,1304.0000,1304.0000,1304.0000
mean,0.1307,3.3141,1.7205
std,0.0789,3.1613,0.8827
min,-0.2150,-26.1300,-5.7300
25%,0.0734,1.7175,1.3400
50%,0.1360,3.5150,1.8700
75%,0.1874,5.2525,2.3000
max,0.3270,11.4500,3.5900


## 2-2. 꼬리 위험 파생 변수 (2개)

- **SKEW_level**: CBOE SKEW Index 원시값  
- **SKEW_zscore**: 63일(약 3개월) 이동 평균/표준편차 기준 Z-score  
  Z > 2이면 꼬리 위험 우려가 역사적으로 매우 높은 상태

In [3]:
# ============================================================
# 꼬리 위험 파생 변수
# ============================================================

# (4) SKEW 원시값
df_feat['SKEW_level'] = df_alt['^SKEW']

# (5) SKEW Z-score: 3개월(63영업일) 롤링 기준
#     평균 대비 몇 표준편차 떨어져 있는지 → 상대적 극단치 감지
roll_mean = df_alt['^SKEW'].rolling(63).mean()
roll_std  = df_alt['^SKEW'].rolling(63).std()
df_feat['SKEW_zscore'] = (df_alt['^SKEW'] - roll_mean) / roll_std

print('=== 꼬리 위험 파생 변수 ===')
df_feat[['SKEW_level', 'SKEW_zscore']].describe().round(4)

=== 꼬리 위험 파생 변수 ===


,SKEW_level,SKEW_zscore
count,1304.0000,1242.0000
mean,140.5501,0.0956
std,13.3129,1.2434
min,110.3400,-3.0705
25%,132.7375,-0.8141
50%,140.9450,0.0848
75%,149.3275,1.0060
max,183.1200,3.8287


## 2-3. 실물 경기 프록시 (2개)

- **Cu_Au_ratio**: 구리/금 가격 비율  
  - 구리 = 산업 수요 민감, 금 = 안전자산 → 비율 상승 = 경기 확장  
- **Cu_Au_ratio_chg**: 21영업일(약 1개월) 변화율  
  - 변화 방향이 실물 경기 모멘텀의 프록시

In [4]:
# ============================================================
# 실물 경기 프록시: Cu/Au 비율
# ============================================================

# 기존 프로젝트의 금 선물(GC=F)과 Step1의 구리 선물(HG=F)
gold_price   = df_prices['GC=F'].reindex(df_alt.index).ffill().bfill()
copper_price = df_alt['HG=F']

# (6) Cu/Au 비율 (원시)
df_feat['Cu_Au_ratio'] = copper_price / gold_price

# (7) Cu/Au 비율 1개월(21일) 변화율
#     상승 → 경기 확장 모멘텀, 하락 → 경기 수축 신호
df_feat['Cu_Au_ratio_chg'] = df_feat['Cu_Au_ratio'].pct_change(21)

print('=== 실물 경기 프록시 ===')
df_feat[['Cu_Au_ratio', 'Cu_Au_ratio_chg']].describe().round(6)

=== 실물 경기 프록시 ===


,Cu_Au_ratio,Cu_Au_ratio_chg
count,1304.000000,1283.000000
mean,0.001952,-0.005044
std,0.000368,0.063753
min,0.001153,-0.238943
25%,0.001658,-0.033813
50%,0.001960,-0.003303
75%,0.002303,0.024868
max,0.002680,0.275488


## 2-4. 신용 시장 파생 변수 (4개)

- **HY_spread**: ICE BofA HY OAS (하이일드 채권과 국채의 금리 차이)  
- **HY_spread_chg**: 5영업일(1주) 변화량 → 급격한 확대 = 자금 경색 신호  
- **yield_curve**: 10Y-2Y Treasury Spread (수익률 곡선 기울기)  
- **yield_curve_inv**: 역전 여부 이진 플래그 (1 = 역전, 0 = 정상)

In [5]:
# ============================================================
# 신용 시장 파생 변수
# ============================================================

# (8) HY 스프레드 원시값 (단위: 퍼센트포인트)
df_feat['HY_spread'] = df_alt['BAMLH0A0HYM2']

# (9) HY 스프레드 1주(5일) 변화량
#     양수 → 스프레드 확대 (신용 리스크 증가)
#     음수 → 스프레드 축소 (신용 환경 개선)
df_feat['HY_spread_chg'] = df_alt['BAMLH0A0HYM2'].diff(5)

# (10) 수익률 곡선 (10Y - 2Y)
#      양수 → 정상적 우상향, 음수 → 역전 (경기침체 선행 신호)
df_feat['yield_curve'] = df_alt['T10Y2Y']

# (11) 수익률 곡선 역전 이진 플래그
#      1 = 역전 (T10Y2Y < 0), 0 = 정상
df_feat['yield_curve_inv'] = (df_alt['T10Y2Y'] < 0).astype(int)

print('=== 신용 시장 파생 변수 ===')
df_feat[['HY_spread', 'HY_spread_chg', 'yield_curve', 'yield_curve_inv']].describe().round(4)


=== 신용 시장 파생 변수 ===


,HY_spread,HY_spread_chg,yield_curve,yield_curve_inv
count,1304.0000,1299.0000,1304.0000,1304.0000
mean,3.6248,-0.0040,0.1672,0.4317
std,0.7002,0.2077,0.6717,0.4955
min,2.5900,-0.7200,-1.0800,0.0000
25%,3.1100,-0.1100,-0.4000,0.0000
50%,3.3800,-0.0100,0.1500,0.0000
75%,4.1025,0.0900,0.5800,1.0000
max,5.9900,1.1100,1.5900,1.0000


## 2-5. 주간 매크로 Nowcasting 파생 변수 (4개)

- **claims_4wma**: 신규 실업수당 4주 이동평균 (노이즈 완화)  
- **claims_zscore**: 52주(1년) 기준 Z-score → 고용 시장 악화 수준  
- **WEI_level**: 주간 경제 활동 지수 원시값 (양수 = 확장, 음수 = 수축)  
- **sahm_indicator**: Sahm 경기침체 지표 (0.50 이상 = 경기침체 시작)

In [6]:
# ============================================================
# 주간 매크로 Nowcasting 파생 변수
# ============================================================

# (12) 신규 실업수당 4주 이동평균 (1주 = 5영업일 → 4주 = 20일)
#      단기 노이즈 완화, 추세 파악용
df_feat['claims_4wma'] = df_alt['ICSA'].rolling(20).mean()

# (13) 신규 실업수당 Z-score (52주 = 260영업일 기준)
#      Z > 2 → 역사적으로 높은 수준 (고용 시장 악화)
claims_52w_mean = df_alt['ICSA'].rolling(260).mean()
claims_52w_std  = df_alt['ICSA'].rolling(260).std()
df_feat['claims_zscore'] = (df_alt['ICSA'] - claims_52w_mean) / claims_52w_std

# (14) Weekly Economic Index 원시값
#      양수 → GDP 성장률 추정치 양호, 음수 → 경기 수축
df_feat['WEI_level'] = df_alt['WEI']

# (15) Sahm 경기침체 지표
#      >= 0.50이면 역사적으로 100% 경기침체 시작 (1970년대 이후)
df_feat['sahm_indicator'] = df_alt['SAHMREALTIME']

print('=== 주간 매크로 Nowcasting ===')
df_feat[['claims_4wma', 'claims_zscore', 'WEI_level', 'sahm_indicator']].describe().round(4)

=== 주간 매크로 Nowcasting ===


,claims_4wma,claims_zscore,WEI_level,sahm_indicator
count,1285.0000,1045.0000,1304.0000,1304.0000
mean,266419.1051,-0.0096,3.0890,0.2483
std,123376.8298,1.0731,2.2081,0.6111
min,197500.0000,-2.4050,-0.9100,-0.3700
25%,214500.0000,-0.8126,1.8300,-0.0300
50%,223750.0000,-0.3032,2.2600,0.1300
75%,237250.0000,0.6513,3.4600,0.3500
max,834200.0000,3.9132,10.5700,3.0300


## 2-6. 파생 변수 전체 요약 + features.csv 저장

In [7]:
# ============================================================
# 파생 변수 전체 요약
# ============================================================

print(f'파생 변수 DataFrame: {df_feat.shape}')
print(f'컬럼 ({len(df_feat.columns)}개): {list(df_feat.columns)}')
print(f'\n--- 결측 현황 (롤링 윈도우 초기값에 의한 자연 결측) ---')
print(df_feat.isnull().sum())

# features.csv 저장
feat_path = os.path.join(DATA_DIR, 'features.csv')
df_feat.to_csv(feat_path)
print(f'\n저장: {feat_path} ({os.path.getsize(feat_path)/1024:.0f} KB)')

파생 변수 DataFrame: (1304, 15)
컬럼 (15개): ['VIX_contango', 'VIX_slope_9d_3m', 'VIX_slope_3m_6m', 'SKEW_level', 'SKEW_zscore', 'Cu_Au_ratio', 'Cu_Au_ratio_chg', 'HY_spread', 'HY_spread_chg', 'yield_curve', 'yield_curve_inv', 'claims_4wma', 'claims_zscore', 'WEI_level', 'sahm_indicator']

--- 결측 현황 (롤링 윈도우 초기값에 의한 자연 결측) ---
VIX_contango         0
VIX_slope_9d_3m      0
VIX_slope_3m_6m      0
SKEW_level           0
SKEW_zscore         62
Cu_Au_ratio          0
Cu_Au_ratio_chg     21
HY_spread            0
HY_spread_chg        5
yield_curve          0
yield_curve_inv      0
claims_4wma         19
claims_zscore      259
WEI_level            0
sahm_indicator       0
dtype: int64

저장: C:\Users\gorhk\DA_Portfolio\01_Daily_Project\PJ12_Asset_Simulater\PJ12_v2_AltData\data\features.csv (273 KB)


## 2-7. df_reg_v2 구축 (확장 회귀 데이터셋)

기존 프로젝트의 `df_reg` 구조를 재현하되, **15개 파생 변수를 추가**합니다.  
기존 구조:
- 타겟: 포트폴리오 실현 변동성 (21일 롤링)
- 기존 피처: 외부 지표 수익률 + 롤링 변동성 + VIX 수준 + FRED 매크로
- 신규 피처: 15개 Tier 1 파생 변수

In [8]:
# ============================================================
# 기존 프로젝트와 동일한 수익률 + 변동성 계산
# ============================================================

# ── 포트폴리오 자산 정의 (기존과 동일) ──
PORT_TICKERS = ['SPY', 'QQQ', 'TLT', 'AGG', 'GLD', 'EEM']
EXT_TICKERS  = ['CL=F', 'GC=F', 'SI=F', 'BTC-USD', 'ETH-USD', '^VIX', 'DX-Y.NYB']

# NYSE 영업일 정렬
nyse_dates = pd.bdate_range(start='2021-01-01', end='2025-12-31', freq='B')
prices = df_prices.reindex(nyse_dates).ffill().bfill()
macro  = df_macro.reindex(nyse_dates).ffill().bfill()

# ── 포트폴리오 일별 수익률 (로그 수익률) ──
df_port_ret = np.log(prices[PORT_TICKERS] / prices[PORT_TICKERS].shift(1)).dropna()

# ── 외부 지표 수익률 ──
#    VIX는 수준 차분 (이미 % 단위이므로 로그 수익률 부적절)
df_ext_ret = pd.DataFrame(index=df_port_ret.index)
for t in EXT_TICKERS:
    if t == '^VIX':
        df_ext_ret[t] = prices[t].diff()  # 수준 차분
    else:
        df_ext_ret[t] = np.log(prices[t] / prices[t].shift(1))  # 로그 수익률
df_ext_ret = df_ext_ret.reindex(df_port_ret.index)

print(f'포트폴리오 수익률: {df_port_ret.shape}')
print(f'외부 지표 수익률 : {df_ext_ret.shape}')

포트폴리오 수익률: (1303, 6)
외부 지표 수익률 : (1303, 7)


In [9]:
# ============================================================
# 포트폴리오 실현 변동성 (21일 롤링, 연율화)
# ============================================================

# 중립형 포트폴리오 동일 비중으로 실현 변동성 계산 (기존 프로젝트 참조)
# 동일 비중(1/6)으로 간략화 — 실제로는 최적 비중 사용
equal_weights = np.array([1/6] * 6)

# 포트폴리오 일별 수익률
port_daily_ret = (df_port_ret * equal_weights).sum(axis=1)

# 21일 롤링 변동성 (연율화: x sqrt(252))
rv_neutral = port_daily_ret.rolling(21).std() * np.sqrt(252)
rv_neutral.name = 'rv_neutral'

print(f'실현 변동성 (rv_neutral): {rv_neutral.dropna().shape[0]}일')
print(f'  평균: {rv_neutral.mean():.4f}')
print(f'  범위: {rv_neutral.min():.4f} ~ {rv_neutral.max():.4f}')

실현 변동성 (rv_neutral): 1283일
  평균: 0.1007
  범위: 0.0413 ~ 0.2745


In [10]:
# ============================================================
# 외부 지표 롤링 변동성 (21일)
# ============================================================

ext_rv = pd.DataFrame(index=df_ext_ret.index)
for t in EXT_TICKERS:
    ext_rv[f'RV_{t}'] = df_ext_ret[t].rolling(21).std() * np.sqrt(252)

print(f'외부 지표 롤링 변동성: {ext_rv.shape}')

외부 지표 롤링 변동성: (1303, 7)


In [11]:
# ============================================================
# df_reg_v2 통합 구축
# ============================================================

# VIX 수준 (기존 df_reg의 VIX_level과 동일)
vix_level = prices['^VIX'].reindex(df_port_ret.index)
vix_level.name = 'VIX_level'

# FRED 매크로 지표 일별 변화
macro_daily = macro[['DGS10']].reindex(df_port_ret.index)
macro_daily['CPI_MoM'] = macro['CPIAUCSL'].pct_change() * 100
macro_daily['UNRATE']  = macro['UNRATE']

# ── 통합 ──
df_reg_v2 = pd.concat([
    # --- 기존 df_reg 구성 요소 (동일하게 유지) ---
    rv_neutral,                     # 타겟: 포트폴리오 실현 변동성
    df_ext_ret,                     # 7개 외부 지표 수익률
    ext_rv,                         # 7개 외부 지표 롤링 변동성
    vix_level,                      # VIX 수준
    macro_daily,                    # FRED 매크로 (DGS10, CPI_MoM, UNRATE)
    # --- 신규: Tier 1 파생 변수 15개 ---
    df_feat.reindex(df_port_ret.index),
], axis=1)

# 롤링 윈도우 초기 NaN 제거
df_reg_v2 = df_reg_v2.dropna()

print(f'df_reg_v2: {df_reg_v2.shape}')
print(f'기간: {df_reg_v2.index[0].date()} ~ {df_reg_v2.index[-1].date()}')
print(f'\n컬럼 ({len(df_reg_v2.columns)}개):')
for i, col in enumerate(df_reg_v2.columns, 1):
    print(f'  {i:2d}. {col}')

df_reg_v2: (1045, 34)
기간: 2021-12-30 ~ 2025-12-31

컬럼 (34개):
   1. rv_neutral
   2. CL=F
   3. GC=F
   4. SI=F
   5. BTC-USD
   6. ETH-USD
   7. ^VIX
   8. DX-Y.NYB
   9. RV_CL=F
  10. RV_GC=F
  11. RV_SI=F
  12. RV_BTC-USD
  13. RV_ETH-USD
  14. RV_^VIX
  15. RV_DX-Y.NYB
  16. VIX_level
  17. DGS10
  18. CPI_MoM
  19. UNRATE
  20. VIX_contango
  21. VIX_slope_9d_3m
  22. VIX_slope_3m_6m
  23. SKEW_level
  24. SKEW_zscore
  25. Cu_Au_ratio
  26. Cu_Au_ratio_chg
  27. HY_spread
  28. HY_spread_chg
  29. yield_curve
  30. yield_curve_inv
  31. claims_4wma
  32. claims_zscore
  33. WEI_level
  34. sahm_indicator


In [12]:
# ============================================================
# df_reg_v2 결측 최종 확인 + CSV 저장
# ============================================================

# 결측 확인 (dropna 이후 0이어야 함)
remaining_na = df_reg_v2.isnull().sum().sum()
print(f'잔여 결측치: {remaining_na}개')

# 저장
reg_path = os.path.join(DATA_DIR, 'df_reg_v2.csv')
df_reg_v2.to_csv(reg_path)

print(f'\n저장: {reg_path}')
print(f'파일 크기: {os.path.getsize(reg_path)/1024:.0f} KB')
print(f'Shape: {df_reg_v2.shape}')
print(f'\n=== 기초 통계량 (주요 신규 변수) ===')
new_cols = ['VIX_contango', 'VIX_slope_9d_3m', 'HY_spread',
            'yield_curve', 'Cu_Au_ratio_chg', 'claims_zscore',
            'WEI_level', 'sahm_indicator']
df_reg_v2[new_cols].describe().round(4)

잔여 결측치: 0개

저장: C:\Users\gorhk\DA_Portfolio\01_Daily_Project\PJ12_Asset_Simulater\PJ12_v2_AltData\data\df_reg_v2.csv
파일 크기: 562 KB
Shape: (1045, 34)

=== 기초 통계량 (주요 신규 변수) ===


,VIX_contango,VIX_slope_9d_3m,HY_spread,yield_curve,Cu_Au_ratio_chg,claims_zscore,WEI_level,sahm_indicator
count,1045.0000,1045.0000,1045.0000,1045.0000,1045.0000,1045.0000,1045.0000,1045.0000
mean,0.1159,2.6842,3.6996,-0.0840,-0.0112,-0.0096,2.4021,0.1821
std,0.0737,2.9581,0.7565,0.4833,0.0605,1.0731,0.9940,0.1931
min,-0.2150,-26.1300,2.5900,-1.0800,-0.2389,-2.4050,1.0600,-0.2700
25%,0.0630,1.3900,3.0900,-0.4500,-0.0386,-0.8126,1.8200,0.0300
50%,0.1214,3.1700,3.5200,-0.1700,-0.0065,-0.3032,2.1700,0.2000
75%,0.1705,4.5000,4.3200,0.3500,0.0209,0.6513,2.5900,0.3500
max,0.3192,8.6200,5.9900,0.8900,0.1755,3.9132,6.3600,0.5700
